In [ ]:
import os
import matplotlib.pyplot as plt
import re
import gzip
import shutil
import pandas as pd
import scipy.sparse
from multiprocessing import Pool
import tarfile
import seaborn as sns
import scanpy.external as sce 
import harmonypy as hm
import anndata as ad
import omicverse as ov
import scvi
import scib
import annoy
import gc
import numpy as np
import random
import scanpy as sc

In [ ]:
sc.settings.verbosity = 1  
sc.logging.print_versions()
sc.settings.set_figure_params(dpi=100, frameon=False, figsize=(4, 4), facecolor='white')
ov.style(font_path='arial')

In [ ]:
os.chdir("")
os.getcwd()

In [ ]:
def save_fig(
    fig_name,
    save_dir="",
    fmt="pdf",
    dpi=300,
):
    save_fig.counter = getattr(save_fig, "counter", 1)
    filename = f"{save_fig.counter:02d}_{fig_name}.{fmt}"
    plt.savefig(
        os.path.join(save_dir, filename),
        format=fmt,
        dpi=dpi,
        bbox_inches="tight",
    )
    save_fig.counter += 1

In [ ]:
adata_xeno = sc.read_h5ad("/20260909GSE2Xeno_GK.h5ad")

In [ ]:
adata_allo = sc.read_h5ad("/20260909AllO_PBMC.h5ad")

In [ ]:
adata_xeno

In [ ]:
adata_xeno.var

In [ ]:
adata_allo

In [ ]:
adata_allo.obs["source"] = "allotransplantation"
adata_allo.obs["species"] = "human"

In [ ]:
adata_merged=sc.concat([adata_xeno,adata_allo],merge='same')

In [ ]:
adata_merged

In [ ]:
adata_merged.var["genesymbol"] = adata_merged.var.index.values

In [ ]:
adata_1 = adata_merged.copy()

In [ ]:
adata_1.obs['sample_8'] = 'allo'
adata_1.obs.loc[adata_1.obs['orig.ident'] == 'Xen_17390399', 'sample_8'] = 'Schmauch'

In [ ]:
adata_2 = adata_1.copy()

In [ ]:
keep_days = ["D-1_batch2","D0_batch2","D0.5_batch1","D1_batch2","D2_batch1",
            "D3_batch1","D7_batch2","D10_batch1","D12_batch3","D14_batch2",
            "D17_batch3","D21_batch2","D25_batch3","D28_batch2","D33_batch3"]

sch = adata_2.obs['sample_8']=='Schmauch'
in_keep = sch & adata_2.obs['batch'].astype(str).isin(keep_days)
adata_2 = adata_2[in_keep | (~sch)].copy()

In [ ]:
adata_2.X.max

In [ ]:
adata_2.var_names_make_unique()
adata_2.obs_names_make_unique()

In [ ]:
sc.pp.filter_cells(adata_2, min_genes=200)
sc.pp.filter_genes(adata_2, min_cells=3)

In [ ]:
adata_2.var["mt"] = adata_2.var_names.str.startswith("MT-")
adata_2.var["ribo"] = adata_2.var_names.str.startswith(("RPS", "RPL"))
adata_2.var["hb"] = adata_2.var_names.str.contains("^HB[^(P)]")

In [ ]:
sc.pp.calculate_qc_metrics(
    adata_2, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True
)

In [ ]:
sc.pl.violin(
    adata_2,
    ["pct_counts_mt", "pct_counts_ribo","pct_counts_hb","n_genes_by_counts", "total_counts"],
    jitter=0,
    multi_panel=True,
    show=False
)
save_fig('figure')
plt.show()

In [ ]:
sc.pl.violin(adata_2, ['n_genes_by_counts','total_counts',
        'pct_counts_mt','pct_counts_ribo','pct_counts_hb'],
        jitter=0, groupby='batch', rotation=45, multi_panel=True)
save_fig('figure')
send_plot()
plt.show()

In [ ]:
with plt.rc_context({'figure.figsize': (20, 4), 'figure.dpi': 300}):
    for metric in ['n_genes_by_counts', 'total_counts',
                    'pct_counts_mt', 'pct_counts_ribo', 'pct_counts_hb']:
        sc.pl.violin(adata_2, metric, jitter=0, groupby='batch',
                    rotation=90, show=False)
        save_fig('figure')
        plt.show()

In [ ]:
import pandas as pd

batch_col = 'batch'
target_batches = ['ABMR_2_pre', 'ABMR_1_pre', 'Normal_1', 'Normal_2']
obs = adata_2.obs

is_target = obs[batch_col].isin(target_batches)

q95_gene = obs.groupby(batch_col)['n_genes_by_counts'].transform(lambda x: x.quantile(0.95))
q95_umi  = obs.groupby(batch_col)['total_counts'].transform(lambda x: x.quantile(0.95))

drop_mask = is_target & (
    (obs['n_genes_by_counts'] > q95_gene) |
    (obs['total_counts'] > q95_umi) |
    (obs['n_genes_by_counts'] < 300) |
    (obs['total_counts'] < 500) |
    (obs['pct_counts_mt'] > 20)
)


stat = pd.DataFrame({
    'Raw': obs.groupby(batch_col).size(),
    'Filtered': drop_mask.groupby(obs[batch_col]).sum(),
    'Pass%': ((1 - drop_mask.groupby(obs[batch_col]).mean()) * 100).round(2)
})
print(stat)

In [ ]:
adata_qc = adata_2[~drop_mask.values].copy()

In [ ]:
adata_qc

In [ ]:
with plt.rc_context({'figure.figsize': (20, 4), 'figure.dpi': 300}):
    for metric in ['n_genes_by_counts', 'total_counts',
                    'pct_counts_mt', 'pct_counts_ribo', 'pct_counts_hb']:
        sc.pl.violin(adata_1, metric, jitter=0, groupby='batch',
                    rotation=90, show=False)
        save_fig('figure')
        plt.show()

In [ ]:
adata_1 = adata_qc.copy()
del adata_qc
gc.collect()

In [ ]:
adata_3 = adata_1.copy()

In [ ]:
adata_3_work = adata_3.copy()

In [ ]:
adata_3.layers["counts"] = adata_3.X.copy()
sc.pp.normalize_total(adata_3, target_sum=1e4)
sc.pp.log1p(adata_3)
adata_3.layers["lognorm"] = adata_3.X.copy()
adata_3.raw = adata_3

In [ ]:
ov.pp.score_genes_cell_cycle(adata_3, species='human')

In [ ]:
sc.pp.highly_variable_genes(
    adata_3,
    flavor="seurat",  
    n_top_genes=3000,
    batch_key="batch",
    subset=False,
)

In [ ]:
ov.pp.regress(adata_3, keys=['S_score', 'G2M_score'], n_jobs=8)

In [ ]:
ov.pp.regress_and_scale(adata_3)

In [ ]:
ov.pp.pca(adata_3, n_pcs=50, layer='regressed_and_scaled',
        mask_var=adata_3.var['highly_variable'].values)
sc.pl.pca_variance_ratio(adata_3, n_pcs=50)

In [ ]:
sc.pp.neighbors(adata_3, n_neighbors=10, use_rep='X_pca', n_pcs=10, key_added='nbr_before_harmony')
sc.tl.umap(adata_3, neighbors_key='nbr_before_harmony')

In [ ]:
import scanpy.external as sce
sce.pp.harmony_integrate(
    adata_3,
    key='batch',
    basis='X_pca',
    adjusted_basis='X_pca_harmony',
    theta=2,
    sigma=0.1,
    max_iter_harmony=30, 
)

In [ ]:
sc.pp.neighbors(adata_3, n_neighbors=10, use_rep='X_pca_harmony', n_pcs=10)
sc.tl.umap(adata_3)

In [ ]:
resolutions = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2]
res_keys = []
for res in resolutions:
    res_keys.append(f"leiden_res_{res:.1f}")
for res, key in zip(resolutions, res_keys):
    sc.tl.leiden(
        adata_3, 
        resolution=res, 
        flavor="igraph", 
        n_iterations=-1, 
        key_added=key
    )

In [ ]:
sc.pl.embedding(
    adata_3,
    basis="umap", 
    color=res_keys, 
    ncols=3,
    size=0.8, 
    frameon=False,
    wspace=0.2,
    hspace=0.3,
    show=False
)
save_fig("")
plt.show()

In [ ]:
adata_3.obs['leiden_auto'] = adata_3.obs['leiden_res_0.8'].astype(str)

In [ ]:
adata_3.obsm["UMAP"] = adata_3.obsm["X_umap"].copy()

In [ ]:
ov.pl.umap(
    adata_3,
    color=["batch","phase",'leiden_auto',"source","orig.ident","species","sample_1","sample_3"],
    frameon='small',
    palette=ov.pl.sc_color,
    ncols=1,
    show=False
)
save_fig('figure')
plt.show()

In [ ]:
with plt.rc_context({'figure.figsize': (20, 4), 'figure.dpi': 300}):
    for metric in ['n_genes_by_counts', 'total_counts',
                    'pct_counts_mt', 'pct_counts_ribo', 'pct_counts_hb']:
        sc.pl.violin(adata_3, metric, jitter=0, groupby='leiden_auto',
                    rotation=90, show=False)
        save_fig('figure')
        plt.show()

In [ ]:
sc.tl.rank_genes_groups(
    adata_3, 
    groupby="leiden_auto", 
    layer="lognorm",  
    use_raw=False,  
    method="wilcoxon", 
    pts=True
)

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata_3, 
    n_genes=3,
    #layer="lognorm",      
    use_raw=False,        
    standard_scale='var', 
    cmap="RdBu_r",
    show=False
)
save_fig("UMAP")
plt.show()

In [ ]:
cluster_counts = adata_3.obs['leiden_auto'].value_counts()
cluster_counts

In [ ]:
top_10_genes = pd.DataFrame(adata_3.uns['rank_genes_groups']['names']).head(10)
top_10_genes

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
print(top_10_genes.head(10).T.to_string()) 

In [ ]:
celltype_coarse = {
    '0': 'Monocytes',
    '1': 'Monocytes',
    '2': 'Monocytes',
    '3': 'Monocytes',
    '4': 'Megakaryocytes',
    '5': 'HSC',
    '6': 'Monocytes',
    '7': 'Monocytes',
    '8': 'Neutrophil',
    '9': 'DC',
    '10': 'Other',
    '11': 'Other',
    '12': 'B_cells',
    '13': 'CD8_T',
    '14': 'NK_cells',
    '15': 'CD8_T',
    '16': 'Other',
    '17': 'Other',
    '18': 'CD8_T',
    '19': 'CD8_T',
    '20': 'CD4_T',
    '21': 'CD8_T',
    '22': 'Megakaryocytes',
}

In [ ]:
adata_3.obs['celltype_coarse'] = adata_3.obs['leiden_auto'].astype(str).map(celltype_coarse)

In [ ]:
ov.pl.umap(
    adata_3,
    color=["celltype_coarse",'leiden_auto'],
    frameon='small',
    palette=ov.pl.sc_color,
    ncols=1,
    show=False
)
save_fig('figure')
plt.show()

In [ ]:
adata_3.obs['celltype_coarse'].value_counts()

Figure 1C

In [ ]:
markers = {
    "NK_cells": ["NKG7", "GNLY", "KLRF1"],
    "CD8_T": ["CD8A", "CD3D", "CD3E"],
    "CD4_T": ["IL7R", "TCF7", "LEF1"],
    "B_cells": ["CD79A", "MS4A1", "CD79B"],
    "Monocytes": ["CD14", "VCAN", "FCN1"],
    "Neutrophil": ["CSF3R", "MMP25", "CMTM2"],
    "Megakaryocytes": ["PPBP", "TUBB1", "GP1BB"],
    'HSC':['CDK6', 'CHRM3', 'NKAIN2'],
    'DC':  ['FCER1A', 'PLD4', 'CD1C'],
}

In [ ]:
marker_final_filtered = {
    k: [g for g in v if g in adata_3.var_names] for k, v in markers.items()
}

In [ ]:
ov.pl.dotplot(
    adata_3,
    marker_final_filtered,
    groupby='celltype_coarse',
    use_raw=True,
    standard_scale='var',
    cmap='Spectral_r',
    #dendrogram=True,
    show=False
)
save_fig('figure')
plt.show()

Figure 1A

In [ ]:
ov.pl.umap(
    adata_3,
    color=["celltype_final",'celltype_coarse_1','leiden_auto',"batch","orig.ident","species","sample_1","sample_3","source"],
    frameon='small',
    palette=ov.pl.sc_color,
    ncols=1,
    show=False
)
save_fig('figure')
plt.show()

Supplementary Fig 1D

In [ ]:
with plt.rc_context({'figure.figsize': (20, 4), 'figure.dpi': 300}):
    for metric in ['n_genes_by_counts', 'total_counts',
                    'pct_counts_mt', 'pct_counts_ribo', 'pct_counts_hb']:
        sc.pl.violin(adata_3, metric, jitter=0, groupby='batch',
                    rotation=90, show=False)
        save_fig('figure')
        plt.show()

In [ ]:
adata_4 = adata_3.copy()

Mono+DC+Megakaryocytes

In [ ]:
adata_4.X = adata_4.layers['counts'].copy()

In [ ]:
adata_mono = adata_4[adata_4.obs['celltype_coarse'].isin(['Monocytes', 'DC','Megakaryocytes'])].copy()

In [ ]:
del adata_mono.obs['leiden_auto']

In [ ]:
cols = [c for c in adata_mono.obs.columns if c.startswith('leiden_res')]
adata_mono.obs = adata_mono.obs.drop(columns=cols)

In [ ]:
adata_mono.X.max()

In [ ]:
adata_mono

In [ ]:
sc.pp.normalize_total(adata_mono, target_sum=1e4)
sc.pp.log1p(adata_mono)
adata_mono.layers['lognorm'] = adata_mono.X.copy()
adata_mono.raw = adata_mono.copy()  

In [ ]:
sc.pp.highly_variable_genes(
    adata_mono, flavor='seurat', n_top_genes=2000,
    batch_key='batch', subset=False,
)

In [ ]:
sc.pp.scale(adata_mono, max_value=10)
sc.tl.pca(adata_mono, n_comps=50, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata_mono, n_pcs=50)

In [ ]:
import scanpy.external as sce
sce.pp.harmony_integrate(
    adata_mono, key='batch', basis='X_pca', adjusted_basis='X_pca_harmony',
    theta=2, sigma=0.1, max_iter_harmony=30,
)

In [ ]:
sc.pp.neighbors(adata_mono, n_neighbors=10, use_rep='X_pca_harmony', n_pcs=10)
sc.tl.umap(adata_mono)
adata_mono.obsm['UMAP'] = adata_mono.obsm['X_umap'].copy() 

In [ ]:
resolutions = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2]
res_keys = []
for res in resolutions:
    res_keys.append(f"leiden_res_{res:.1f}")
for res, key in zip(resolutions, res_keys):
    sc.tl.leiden(
        adata_mono, 
        resolution=res, 
        flavor="igraph", 
        n_iterations=-1, 
        key_added=key
    )

In [ ]:
sc.pl.embedding(
    adata_mono,
    basis="UMAP", 
    color=res_keys,
    ncols=3,
    size=0.8,  
    frameon=False,
    wspace=0.2,
    hspace=0.3,
    show=False
)
save_fig("")
plt.show()

In [ ]:
adata_mono.obs['leiden_auto'] = adata_mono.obs['leiden_res_0.6'].astype(str)

In [ ]:
ov.pl.umap(
    adata_mono,
    color=["batch","phase",'leiden_auto',"source","orig.ident","species","sample_1","sample_3"],
    frameon='small',
    palette=ov.pl.sc_color,
    ncols=1,
    show=False
)
save_fig("UMAP亚群")
plt.show()

In [ ]:
with plt.rc_context({'figure.figsize': (20, 4), 'figure.dpi': 300}):
    for metric in ['n_genes_by_counts', 'total_counts',
                    'pct_counts_mt', 'pct_counts_ribo', 'pct_counts_hb']:
        sc.pl.violin(adata_mono, metric, jitter=0, groupby='leiden_auto',
                    rotation=90, show=False)
        save_fig('figure')
        plt.show()

In [ ]:
sc.tl.rank_genes_groups(
    adata_mono, 
    groupby="leiden_auto", 
    layer="lognorm",  
    use_raw=False,  
    method="wilcoxon", 
    pts=True
)

In [ ]:
sc.tl.dendrogram(adata_mono, groupby='leiden_auto')
sc.pl.rank_genes_groups_dotplot(
    adata_mono, 
    n_genes=3,  
    use_raw=False,        
    standard_scale='var',
    cmap="RdBu_r",
    show=False
)
save_fig("UMAP亚群")
plt.show()

In [ ]:
cluster_counts = adata_mono.obs['leiden_auto'].value_counts()
cluster_counts

In [ ]:
top_10_genes = pd.DataFrame(adata_mono.uns['rank_genes_groups']['names']).head(10)
top_10_genes

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
print(top_10_genes.head(10).T.to_string()) 

In [ ]:
celltype_fine = {
    '0': 'Monocytes',
    '1': 'Monocytes',
    '2': 'Monocytes',
    '3': 'DC',
    '4': 'Monocytes',
    '5': 'Monocytes',
    '6': 'Monocytes',
    '7': 'Monocytes',
    '8': 'Monocytes',
    '9': 'Other',
    '10': 'Other',
    '11': 'Monocytes',
    '12': 'DC',
    '13': 'Megakaryocytes',
    '14': 'Other',
}

In [ ]:
celltype_2fine = {
    '0': 'Non_classical_Monocytes',
    '1': 'Classical_Monocytes',
    '2': 'Classical_Monocytes',
    '3': 'cDC2',
    '4': 'Classical_Monocytes',
    '5': 'Classical_Monocytes',
    '6': 'Classical_Monocytes',
    '7': 'Classical_Monocytes',
    '8': 'Classical_Monocytes',
    '9': 'Other',
    '10': 'Other',
    '11': 'Classical_Monocytes',
    '12': 'pDC',
    '13': 'Megakaryocytes',
    '14': 'Other',
}

In [ ]:
adata_mono.obs['celltype_fine'] = adata_mono.obs['leiden_auto'].astype(str).map(celltype_fine)

In [ ]:
adata_mono.obs['celltype_2fine'] = adata_mono.obs['leiden_auto'].astype(str).map(celltype_2fine)

In [ ]:
ov.pl.umap(
    adata_mono,
    color=["celltype_fine","celltype_2fine",'leiden_auto',"batch","orig.ident","species","sample_1","sample_3","source"],
    frameon='small',
    palette=ov.pl.sc_color,
    ncols=1,
    show=False
)
save_fig('figure')
plt.show()

In [ ]:
markers = {
    'Classical_mono':   ['S100A12','CD14','LYZ','FCN1'],
    'Nonclassical_mono':   ['FCGR3A','CDKN1C'],
    'pDC':  ['LILRA4', 'CLEC4C', 'SERPINF1', 'IL3RA'],
    'cDC2': ['CD1C', 'FCER1A', 'CLEC10A', 'CD1E'],
    "Megakaryocytes": ["PPBP", "TUBB1", "GP1BB"],
}


In [ ]:
# 只留基因空间里有的
markers = {k: [g for g in v if g in adata_mono.var_names]
                for k, v in markers.items()}


In [ ]:
ov.pl.dotplot(
    adata_mono,
    markers,
    groupby='celltype_2fine', 
    use_raw=True,
    standard_scale='var',
    cmap='Spectral_r',
    dendrogram=True,
)
save_fig('figure')
plt.show()

T_cells and NK_cells

In [ ]:
adata_t = adata_4[adata_4.obs['celltype_coarse'].isin(['CD8_T', 'NK_cells','CD4_T'])].copy()

In [ ]:
del adata_t.obs['leiden_auto']

In [ ]:
cols = [c for c in adata_t.obs.columns if c.startswith('leiden_res')]
adata_t.obs = adata_t.obs.drop(columns=cols)

In [ ]:
adata_t.X.max()

In [ ]:
adata_t

In [ ]:
sc.pp.normalize_total(adata_t, target_sum=1e4)
sc.pp.log1p(adata_t)
adata_t.layers['lognorm'] = adata_t.X.copy()
adata_t.raw = adata_t.copy() 

In [ ]:
sc.pp.highly_variable_genes(
    adata_t, flavor='seurat', n_top_genes=2000,
    batch_key='batch', subset=False,
)

In [ ]:
sc.pp.scale(adata_t, max_value=10)
sc.tl.pca(adata_t, n_comps=50, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata_t, n_pcs=50)

In [ ]:
import scanpy.external as sce
sce.pp.harmony_integrate(
    adata_t, key='batch', basis='X_pca', adjusted_basis='X_pca_harmony',
    theta=2, sigma=0.1, max_iter_harmony=30,
)

In [ ]:
sc.pp.neighbors(adata_t, n_neighbors=8, use_rep='X_pca_harmony', n_pcs=8)
sc.tl.umap(adata_t)

In [ ]:
resolutions = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2]
res_keys = []
for res in resolutions:
    res_keys.append(f"leiden_res_{res:.1f}")
for res, key in zip(resolutions, res_keys):
    sc.tl.leiden(
        adata_t, 
        resolution=res, 
        flavor="igraph", 
        n_iterations=-1, 
        key_added=key
    )

In [ ]:
sc.pl.embedding(
    adata_t,
    basis="UMAP", 
    color=res_keys, 
    ncols=3,
    size=0.8, 
    frameon=False,
    wspace=0.2,
    hspace=0.3,
    show=False
)
save_fig("UMAP")
plt.show()

In [ ]:
adata_t.obs['leiden_auto'] = adata_t.obs['leiden_res_1.0'].astype(str)

In [ ]:
ov.pl.umap(
    adata_t,
    color=["batch","phase",'leiden_auto',"source","orig.ident","species","sample_1","sample_3"],
    frameon='small',
    palette=ov.pl.sc_color,
    ncols=1,
    show=False
)
save_fig('figure')
plt.show()

In [ ]:
with plt.rc_context({'figure.figsize': (20, 4), 'figure.dpi': 300}):
    for metric in ['n_genes_by_counts', 'total_counts',
                    'pct_counts_mt', 'pct_counts_ribo', 'pct_counts_hb']:
        sc.pl.violin(adata_t, metric, jitter=0, groupby='leiden_auto',
                    rotation=90, show=False)
        save_fig('figure')
        plt.show()

In [ ]:
sc.tl.rank_genes_groups(
    adata_t, 
    groupby="leiden_auto", 
    layer="lognorm",  
    use_raw=False,  
    method="wilcoxon", 
    pts=True
)

In [ ]:
sc.tl.dendrogram(adata_t, groupby='leiden_auto')
sc.pl.rank_genes_groups_dotplot(
    adata_t, 
    n_genes=5,
    #layer="lognorm",      
    use_raw=False,        
    standard_scale='var', 
    cmap="RdBu_r"
)
plt.show()

In [ ]:
cluster_counts = adata_t.obs['leiden_auto'].value_counts()
cluster_counts

In [ ]:
top_10_genes = pd.DataFrame(adata_t.uns['rank_genes_groups']['names']).head(10)
top_10_genes

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
print(top_10_genes.head(10).T.to_string()) 

In [ ]:
markers = {
    "T_cells": ["CD3G", "CD3E", "CD3D"],
    "Naive": ["LEF1", "TCF7", "SELL",  "CCR7"],
    "CD4_T": ["CD4", "TSHZ2"],
    "CD8_T": ["CD8A", "CD8B"],
    'CD8_Tem': ['GZMK', 'KLRG1', 'CST7'],
    'CD4_Tem': ['LTB', 'IL7R', 'S100A4'],
    'CD8_Temra': ['NKG7', 'GZMH', 'ZEB2', 'FGFBP2'],
    "MAIT": ["SLC4A10", "KLRB1", "RAV1-2"],
    "Proliferating_cell": ["TOP2A", "TYMS", "MKI67"],
    "NK_cells": ["NKG7", "GNLY", "KLRF1","CD56"],
}

In [ ]:
markers = {k: [g for g in v if g in adata_t.var_names]
                for k, v in markers.items()}


In [ ]:
ov.pl.dotplot(
    adata_t,
    markers,
    groupby='leiden_auto',
    use_raw=True,
    standard_scale='var',
    cmap='Spectral_r',
    dendrogram=True,
)
plt.show()

In [ ]:
celltype_fine = {
    '0':  'NK_cells',
    '1':  'CD8_T',
    '2':  'NK_cells',
    '3':  'CD8_T',
    '4':  'CD8_T',
    '5':  'CD4_T',
    '6':  'Other',
    '7':  'NK_cells',
    '8':  'CD4_T',
    '9':  'CD8_T',
    '10': 'CD8_T',
    '11': 'CD4_T',
    '12': 'CD4_T',
    '13': 'CD4_T',
    '14': 'Other',
    '15': 'CD8_T',
    '16': 'Other',
    '17': 'NK_cells',
    '18': 'CD4_T',
    '19': 'CD8_T',
    '20': 'CD8_T',
    '21': 'NK_cells',
    '22': 'Other',
}

In [ ]:
celltype_2fine = {
    '0':  'CD16+ NK_cells',
    '1':  'CD8_Temra',
    '2':  'CD16+ NK_cells',
    '3':  'CD8_Temra',
    '4':  'CD8_Temra',
    '5':  'CD4_Naive',
    '6':  'Proliferating_cell',
    '7':  'CD16- NK_cells',
    '8':  'CD4_Naive',
    '9':  'CD8_Naive',
    '10': 'CD8_Tem',
    '11': 'CD4_Naive',
    '12': 'CD4_Tcm',
    '13': 'CD4_Tcm',
    '14': 'Other',
    '15': 'CD8_Temra',
    '16': 'Other',
    '17': 'CD16+ NK_cells',
    '18': 'CD4_Naive',
    '19': 'MAIT',
    '20': 'CD8_Tem',
    '21': 'CD16+ NK_cells',
    '22': 'Other',
}

In [ ]:
adata_t.obs['celltype_fine'] = adata_t.obs['leiden_auto'].astype(str).map(celltype_fine)

In [ ]:
adata_t.obs['celltype_2fine'] = adata_t.obs['leiden_auto'].astype(str).map(celltype_2fine)

In [ ]:
ov.pl.umap(
    adata_t,
    color=["celltype_fine","celltype_2fine",'leiden_auto',"batch","orig.ident","species","sample_1","sample_3","source"],
    frameon='small',
    palette=ov.pl.sc_color,
    ncols=1,
    show=False
)
save_fig('figure')
plt.show()

In [ ]:
markers = {
    'Naive':             ['LEF1', 'TCF7', 'SELL', 'CCR7'],
    'CD4':               ['CD4', 'TSHZ2'],
    'CD8':               ['CD8A', 'CD8B'],
    'CD8_Tem':           ['GZMK', 'IL7R', 'THEMIS'],
    'CD8_Temra':         ['GZMB', 'GZMH', 'PRF1', 'ZEB2', 'NKG7', 'FGFBP2', 'KLRG1', 'CST7'],
    'CD4_Tcm':           ['LTB', 'IL7R'],
    'MAIT':              ['SLC4A10', 'KLRB1', 'RORA', 'ZBTB16'],
    'CD16+ NK':          ['FCGR3A', 'SPON2', 'FGFBP2', 'TYROBP'],
    'CD16- NK':          ['NCAM1', 'KLRC1', 'IL2RB'],
    'Proliferating_cell': ['TOP2A', 'MKI67', 'TYMS'],
}


In [ ]:
markers = {
    'Naive':      ['LEF1', 'TCF7', 'SELL', 'CCR7'],
    'CD4':        ['CD4', 'TSHZ2'],
    'CD8':        ['CD8A', 'CD8B'],
    'CD4_Tcm':    ['LTB', 'IL7R'],
    'CD8_Tem':    ['GZMK', 'IL7R'],
    'CD8_Temra':  ['GZMB',  'PRF1', 'ZEB2'],
    'MAIT':       ['SLC4A10', 'KLRB1', 'RORA', 'ZBTB16'],
    'NK':         ['GNLY', 'KLRF1', 'FCGR3A', 'NCAM1'],
    'Proliferating_cell': ['TOP2A', 'MKI67', 'TYMS'],
}


In [ ]:
markers = {k: [g for g in v if g in adata_t.var_names]
                for k, v in markers.items()}


In [ ]:
ov.pl.dotplot(
    adata_t,
    markers,
    groupby='celltype_2fine',   
    use_raw=True,
    standard_scale='var',
    cmap='Spectral_r',
    dendrogram=True,
)
save_fig('figure')
plt.show()

In [ ]:
ov.pl.dotplot(
    adata_t,
    markers,
    groupby='celltype_fine', 
    use_raw=True,
    standard_scale='var',
    cmap='Spectral_r',
    dendrogram=True,
)
save_fig('figure')
plt.show()

In [ ]:
sc.pl.dotplot(
    adata_t,
    markers,
    groupby='celltype_2fine', 
    use_raw=True,
    standard_scale='var',
    cmap='RdBu_r',
    dendrogram=False,
    show=False
)
save_fig('figure')
plt.show()

B_cells

In [ ]:
adata_b = adata_4[adata_4.obs['celltype_coarse'].isin(['B_cells'])].copy()

In [ ]:
del adata_b.obs['leiden_auto']

In [ ]:
cols = [c for c in adata_b.obs.columns if c.startswith('leiden_res')]
adata_b.obs = adata_b.obs.drop(columns=cols)

In [ ]:
adata_b.X.max()

In [ ]:
adata_b

In [ ]:
sc.pp.normalize_total(adata_b, target_sum=1e4)
sc.pp.log1p(adata_b)
adata_b.layers['lognorm'] = adata_b.X.copy()
adata_b.raw = adata_b.copy() 

In [ ]:
sc.pp.highly_variable_genes(
    adata_b, flavor='seurat', n_top_genes=2000,
    batch_key='batch', subset=False,
)

In [ ]:
sc.pp.scale(adata_b, max_value=10)
sc.tl.pca(adata_b, n_comps=50, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata_b, n_pcs=50)

In [ ]:
import scanpy.external as sce
sce.pp.harmony_integrate(
    adata_b, key='batch', basis='X_pca', adjusted_basis='X_pca_harmony',
    theta=2, sigma=0.1, max_iter_harmony=30,
)

In [ ]:
sc.pp.neighbors(adata_b, n_neighbors=8, use_rep='X_pca_harmony', n_pcs=8)
sc.tl.umap(adata_b)

In [ ]:
resolutions = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2]
res_keys = []
for res in resolutions:
    res_keys.append(f"leiden_res_{res:.1f}")
for res, key in zip(resolutions, res_keys):
    sc.tl.leiden(
        adata_b, 
        resolution=res, 
        flavor="igraph", 
        n_iterations=-1, 
        key_added=key
    )

In [ ]:
sc.pl.embedding(
    adata_b,
    basis="UMAP", 
    color=res_keys, 
    ncols=3,
    size=0.8,  
    frameon=False,
    wspace=0.2,
    hspace=0.3,
    show=False
)
save_fig("UMAP")
send_plot()
plt.show()

In [ ]:
adata_b.obs['leiden_auto'] = adata_b.obs['leiden_res_0.4'].astype(str)

In [ ]:
ov.pl.umap(
    adata_b,
    color=["batch","phase",'leiden_auto',"source","orig.ident","species","sample_1","sample_3"],
    frameon='small',
    palette=ov.pl.sc_color,
    ncols=1,
    show=False
)
save_fig("UMAP亚群")
plt.show()

In [ ]:
with plt.rc_context({'figure.figsize': (20, 4), 'figure.dpi': 300}):
    for metric in ['n_genes_by_counts', 'total_counts',
                    'pct_counts_mt', 'pct_counts_ribo', 'pct_counts_hb']:
        sc.pl.violin(adata_b, metric, jitter=0, groupby='leiden_auto',
                    rotation=90, show=False)
        save_fig('figure')
        plt.show()

In [ ]:
sc.tl.rank_genes_groups(
    adata_b, 
    groupby="leiden_auto", 
    layer="lognorm",  
    use_raw=False,  
    method="wilcoxon", 
    pts=True
)

In [ ]:
sc.tl.dendrogram(adata_b, groupby='leiden_auto')
sc.pl.rank_genes_groups_dotplot(
    adata_b, 
    n_genes=5,
    #layer="lognorm",      
    use_raw=False,        
    standard_scale='var',
    cmap="RdBu_r"
)
plt.show()

In [ ]:
cluster_counts = adata_b.obs['leiden_auto'].value_counts()
cluster_counts

In [ ]:
top_10_genes = pd.DataFrame(adata_b.uns['rank_genes_groups']['names']).head(10)
top_10_genes

In [ ]:
celltype_fine = {
    '0': 'B_cell',
    '1': 'B_cell',
    '2': 'B_cell',
    '3': 'B_cell',
    '4': 'Plasma_cell',
    '5': 'B_cell',
    '6': 'B_cell',
    '7': 'Other',
    '8': 'Other',
}

In [ ]:
celltype_2fine = {
    '0': 'Memory_B',
    '1': 'Memory_B',
    '2': 'Naive_B',
    '3': 'Memory_B',
    '4': 'Plasma_cell',
    '5': 'Naive_B',
    '6': 'Naive_B',
    '7': 'Other',
    '8': 'Other',
}

In [ ]:
adata_b.obs['celltype_fine'] = adata_b.obs['leiden_auto'].astype(str).map(celltype_fine)

In [ ]:
adata_b.obs['celltype_2fine'] = adata_b.obs['leiden_auto'].astype(str).map(celltype_2fine)

In [ ]:
ov.pl.umap(
    adata_b,
    color=["celltype_fine","celltype_2fine",'leiden_auto',"batch","orig.ident","species","sample_1","sample_3","source"],
    frameon='small',
    palette=ov.pl.sc_color,
    ncols=1,
    show=False
)
save_fig("UMAP")
plt.show()

In [ ]:
markers = {
    'Naive_B': ['IGHD', 'IGHM', 'TCL1A', 'IL4R'],
    'Plasma_cell': ['JCHAIN', 'MZB1', 'ITM2C'],
}


In [ ]:
markers = {k: [g for g in v if g in adata_b.var_names]
                for k, v in markers.items()}


In [ ]:
ov.pl.dotplot(
    adata_b,
    markers,
    groupby='celltype_2fine', 
    use_raw=True,
    standard_scale='var',
    cmap='Spectral_r',
    dendrogram=True,
)
save_fig('figure')
plt.show()

In [ ]:
adata_5 = adata_4.copy()

In [ ]:
adata_5.obs['celltype_coarse_1'] = adata_5.obs['celltype_coarse'].astype(str)
adata_5.obs['celltype_final'] = adata_5.obs['celltype_coarse'].astype(str)

In [ ]:
for name, sub in [('adata_b', adata_b),
                ('adata_t', adata_t),
                ('adata_mono', adata_mono)]:
    hit = adata_5.obs_names.isin(sub.obs_names)
    fine = sub.obs['celltype_fine'].astype(str)
    two = sub.obs['celltype_2fine'].astype(str)
    adata_5.obs.loc[hit, 'celltype_coarse_1'] = adata_5.obs_names[hit].map(fine).values
    adata_5.obs.loc[hit, 'celltype_final'] = adata_5.obs_names[hit].map(two).values
    print(f'{name}: 回填 {hit.sum()} / 亚群 {sub.n_obs}')

Figure 1C

In [ ]:
markers = {
    "NK_cells": ["NKG7", "GNLY", "KLRF1"],
    "CD8_T": ["CD8A", "CD3D", "CD3E"],
    "CD4_T": ["IL7R", "TCF7", "LEF1"],
    "B_cell": ["CD79A", "MS4A1", "CD79B"],
    "Monocytes": ["CD14", "VCAN", "FCN1"],
    "Neutrophil": ["CSF3R", "MMP25", "CMTM2"],
    "Megakaryocytes": ["PPBP", "TUBB1", "GP1BB"],
    'HSC':['CDK6', 'CHRM3', 'NKAIN2'],
    'DC':  ['FCER1A', 'PLD4', 'CD1C'],
}

In [ ]:
marker_final_filtered = {
    k: [g for g in v if g in adata_5.var_names] for k, v in markers.items()
}

In [ ]:
ov.pl.dotplot(
    adata_5,
    marker_final_filtered,
    groupby='celltype_final', 
    use_raw=True,
    standard_scale='var',
    cmap='Spectral_r',
    #dendrogram=True,
    show=False
)
save_fig('figure')
plt.show()

Supplementary Fig 1A&B

In [ ]:
ov.pl.umap(
    adata_5,
    color=["celltype_final",'celltype_coarse_1','leiden_auto',"batch","orig.ident","species","sample_1","sample_3","source"],
    frameon='small',
    palette=ov.pl.sc_color,
    ncols=1,
    show=False
)
save_fig('figure')
plt.show()

Supplementary Fig 1C

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ov.pl.cellproportion(adata=adata_5, celltype_clusters='celltype_final', groupby='batch', legend=False, ax=ax, show=False)
ax.legend(*ax.get_legend_handles_labels(), bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
save_fig('figure')
plt.show()

Supplementary Fig 1D

In [ ]:
with plt.rc_context({'figure.figsize': (20, 4), 'figure.dpi': 300}):
    for metric in ['n_genes_by_counts', 'total_counts',
                    'pct_counts_mt', 'pct_counts_ribo', 'pct_counts_hb']:
        sc.pl.violin(adata_5, metric, jitter=0, groupby='batch',
                    rotation=90, show=False)
        save_fig('figure')
        plt.show()

Fig 1D

In [ ]:
fig, ax = plt.subplots(figsize=(3, 5))
ov.pl.cellproportion(adata=adata_5, celltype_clusters='celltype_coarse', groupby='source', legend=False, ax=ax, show=False)
ax.legend(*ax.get_legend_handles_labels(), bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
save_fig('figure')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ov.pl.cellproportion(adata=adata_5, celltype_clusters='celltype_final', groupby='batch', legend=False, ax=ax, show=False)
ax.legend(*ax.get_legend_handles_labels(), bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
save_fig('figure')
plt.show()

Fig 1B

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import pymn
import seaborn as sns

%matplotlib inline

matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42
sns.set(style="white", font_scale=1.25)
plt.rc("axes.spines", top=False, right=False)

In [ ]:
adata_ovb = adata_5.copy()

In [ ]:
pymn.variableGenes(adata_ovb, study_col="orig.ident")
var_genes = adata_ovb.var_names[adata_ovb.var["highly_variable"]].astype(str)
clean_var_genes = var_genes[
    ~var_genes.str.startswith(("MT-", "RPL", "RPS"))
    & ~var_genes.isin(["HBA1", "HBA2", "HBB", "HBD"])
].tolist()

In [ ]:
pymn.MetaNeighborUS(
    adata_ovb,
    study_col="orig.ident",
    ct_col="celltype_coarse",
    var_genes=clean_var_genes,
    fast_version=True,
    symmetric_output=False,
    one_vs_best=True,
)

In [ ]:
display(pd.DataFrame(adata_ovb.uns["MetaNeighborUS_1v1"]).round(3))

In [ ]:
pymn.plotMetaNeighborUS(
    adata_ovb,
    mn_key="MetaNeighborUS_1v1",
    cmap="coolwarm",
    figsize=(10, 10),
    xticklabels=True,
    yticklabels=True,
    fontsize=8,
    show=False,
)
save_fig("MetaNeighborUS")
plt.show()